# Hugging Face Transformers: Pipeline 기초 (VSCode 실습용)

---

이 노트북은 Hugging Face **Transformers** 라이브러리의 **pipeline** 기능을 학습합니다.
`pipeline`을 사용하면 복잡한 딥러닝 모델을 **단 몇 줄의 코드**로 활용할 수 있습니다.

### 학습 로드맵

| 순서 | 주제 | 핵심 내용 |
|------|------|-----------|
| 0 | Transformers와 Pipeline이란? | 라이브러리 개요, Pipeline 개념 |
| 1 | VSCode 환경 설정 | 가상환경, 패키지 설치 |
| 2 | 라이브러리 불러오기 | import, 버전 확인 |
| 3 | Pipeline 동작 방식 | Task 정의, 모델 지정, 내부 처리 흐름 |
| 4 | 예제 1: 감성 분석 | 텍스트 긍정/부정 분류 |
| 5 | 예제 2: 번역 | 한국어 ↔ 영어 번역 |
| 6 | 예제 3: 이미지 분류 | 이미지 카테고리 분류 |
| 7 | 예제 4: 객체 탐지 | 이미지 내 물체 감지 + 바운딩 박스 |
| - | Quick Reference | 핵심 문법 요약표 |

## 0. Transformers와 Pipeline이란?

### Transformers 라이브러리

**Hugging Face Transformers**는 Transformer 기반 딥러닝 모델을 간편하게 활용할 수 있는 **오픈소스 라이브러리**입니다.

| 항목 | 설명 |
|------|------|
| 개발사 | Hugging Face |
| 지원 모델 | BERT, GPT, T5, ViT, DETR 등 **수만 개** |
| 지원 태스크 | 텍스트 분류, 번역, 요약, 이미지 분류, 객체 탐지 등 |
| 모델 저장소 | [Hugging Face Hub](https://huggingface.co/models) — 사전학습된 모델 검색·다운로드 |
| 최신 버전 | v5.x (2026년 1월 출시, 5년 만의 메이저 업데이트) |

### Pipeline이란?

`pipeline`은 **모델 로드 → 전처리 → 추론 → 후처리** 과정을 **한 줄**로 캡슐화한 고수준 API입니다.

```
입력 데이터 → [전처리(Tokenizer/Processor)] → [모델 추론] → [후처리] → 결과
```

- **장점**: 코드가 매우 간단 — 초보자도 즉시 사용 가능
- **단점**: 세밀한 제어가 어렵고, 한국어 등 비영어권에서 정확도가 낮을 수 있음

### 주요 Task 목록

| Task 이름 | 설명 | 기본 모델 |
|-----------|------|-----------|
| `text-classification` | 텍스트 감성 분석 (긍정/부정) | distilbert-base-uncased-finetuned-sst-2-english |
| `question-answering` | 질문에 대한 답변 추출 | distilbert-base-uncased-distilled-squad |
| `text-generation` | 텍스트 생성 | gpt2 |
| `translation` | 번역 (v5에서 제거) | 모델 직접 지정 필요 |
| `summarization` | 요약 (v5에서 제거) | facebook/bart-large-cnn |
| `ner` | 개체명 인식 | dbmdz/bert-large-cased-finetuned-conll03-english |
| `image-classification` | 이미지 분류 | google/vit-base-patch16-224 |
| `object-detection` | 객체 탐지 | facebook/detr-resnet-50 |

> **참고 — Transformers v5 주요 변경사항**:
> - `TranslationPipeline`, `SummarizationPipeline` 제거 → `text-generation` + LLM 사용 권장
> - `AutoFeatureExtractor` → `AutoImageProcessor` 로 통합
> - 매주 마이너 버전 릴리스 (v5.1, v5.2, ...)
> - 이 노트북의 번역 예제는 v4 기준이며, v5에서는 별도 안내를 따라 주세요.

## 1. VSCode 환경 설정 (가상환경 및 패키지 설치)

VSCode에서 파이썬을 실행하기 전, **가상환경(.venv)**을 만들고 필요한 패키지를 설치합니다.

### 1단계: 가상환경 생성 (터미널에서 실행)

```bash
# 프로젝트 폴더로 이동
cd my_project

# 가상환경 생성
python -m venv .venv

# 가상환경 활성화
# Windows:
.venv\Scripts\activate
# macOS/Linux:
source .venv/bin/activate
```

### 2단계: 패키지 설치

```bash
# Transformers + PyTorch (CPU 버전)
pip install transformers torch torchvision

# 이미지 처리 및 시각화
pip install Pillow matplotlib requests

# 번역 모델(NLLB)에 필요한 추가 패키지
pip install sentencepiece protobuf
```

> **버전 참고**: 이 노트북의 번역 예제는 Transformers v4 기준입니다.
> v5에서는 `TranslationPipeline`이 제거되었으므로, 번역 예제 실행이 필요하면
> `pip install 'transformers>=4.45,<5'` 로 v4를 설치하세요.

### 3단계: VSCode 커널 선택

1. `.ipynb` 파일 열기
2. 우측 상단 **커널 선택** 클릭
3. `.venv (Python 3.x.x)` 선택

> **참고**: 모델을 처음 사용할 때 Hugging Face Hub에서 **자동 다운로드**됩니다.
> 인터넷 연결이 필요하며, 모델 크기에 따라 시간이 걸릴 수 있습니다.
> 다운로드된 모델은 `~/.cache/huggingface/` 에 캐시됩니다.

## 2. 라이브러리 불러오기 및 환경 확인

Transformers와 PyTorch를 불러오고, 설치된 버전과 GPU 사용 가능 여부를 확인합니다.

In [ ]:
# ──────────────────────────────────────────────
# 라이브러리 불러오기 및 환경 확인
# ──────────────────────────────────────────────
import transformers
import torch

print(f"Transformers 버전: {transformers.__version__}")
print(f"PyTorch 버전:      {torch.__version__}")
print(f"CUDA 사용 가능:    {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 이름:          {torch.cuda.get_device_name(0)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스:     {device}")

## 3. Pipeline 동작 방식

### 사용 방법 2가지

Pipeline을 사용하는 방법은 크게 **두 가지**입니다.

#### 방법 1: Task만 지정 (가장 간단)

```python
pipe = pipeline(task="sentiment-analysis")   # 기본 모델 자동 선택
result = pipe("I love this!")                 # 바로 추론
```

- `task`만 지정하면, 내부 매핑 테이블에서 **기본 모델**을 자동 선택
- 빠르게 테스트할 때 유용하지만, 한국어 등에서는 정확도가 낮을 수 있음

#### 방법 2: 모델 직접 지정 (권장)

```python
pipe = pipeline(
    task="sentiment-analysis",
    model="sangrimlee/bert-base-multilingual-cased-nsmc",
    tokenizer="sangrimlee/bert-base-multilingual-cased-nsmc"
)
result = pipe("이 영화 정말 좋았어요!")
```

- 특정 언어나 도메인에 맞는 **미세조정(Fine-tuned) 모델** 사용 가능
- 모델과 토크나이저(전처리기)를 분리하여 로드할 수도 있음

### Pipeline 내부 처리 흐름

```
pipeline("sentiment-analysis")("이 영화 좋았어요!")
    │
    ├─ 1. 전처리 (Tokenizer)
    │     "이 영화 좋았어요!" → 토큰 ID: [101, 9638, 16439, ..., 102]
    │
    ├─ 2. 모델 추론 (Model)
    │     토큰 ID → logits: [-3.12, 2.87]
    │
    └─ 3. 후처리 (Post-processing)
          logits → softmax → {"label": "POSITIVE", "score": 0.95}
```

### 모델 직접 로드 방식 (Pipeline 없이)

Pipeline을 사용하지 않고 직접 구현할 수도 있습니다:

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

encoded = tokenizer("입력 텍스트", return_tensors="pt")

with torch.no_grad():               # 추론 시 gradient 계산 비활성화
    outputs = model(**encoded)
    logits  = outputs.logits         # 모델의 raw 출력값

probs = torch.softmax(logits, dim=-1)  # 확률로 변환
```

> Pipeline은 위 과정을 **자동화**한 것입니다.
> 세밀한 제어가 필요할 때는 직접 로드 방식을 사용하세요.

### Task별 Auto 클래스 매핑

| Task | Auto 모델 클래스 | Auto 전처리 클래스 |
|------|------------------|---------------------|
| `text-classification` | AutoModelForSequenceClassification | AutoTokenizer |
| `question-answering` | AutoModelForQuestionAnswering | AutoTokenizer |
| `text-generation` | AutoModelForCausalLM | AutoTokenizer |
| `ner` | AutoModelForTokenClassification | AutoTokenizer |
| `image-classification` | AutoModelForImageClassification | AutoImageProcessor |
| `object-detection` | AutoModelForObjectDetection | AutoImageProcessor |

## 4. 예제 1: 감성 분석 (Sentiment Analysis)

텍스트가 **긍정(POSITIVE)**인지 **부정(NEGATIVE)**인지 분류하는 태스크입니다.

| 구분 | 내용 |
|------|------|
| Task 이름 | `sentiment-analysis` (= `text-classification`) |
| 기본 모델 | distilbert-base-uncased-finetuned-sst-2-english |
| 한국어 모델 | sangrimlee/bert-base-multilingual-cased-nsmc |
| 출력 | label (POSITIVE/NEGATIVE), score (확률) |

> **주의**: 기본 모델은 **영어 전용**이므로, 한국어 입력 시 정확도가 매우 낮습니다.
> 한국어 분석에는 **한국어로 미세조정된 모델**을 직접 지정해야 합니다.

In [ ]:
# ──────────────────────────────────────────────
# 감성 분석 (1): 기본 pipeline — task만 지정
#   기본 모델: distilbert-base-uncased-finetuned-sst-2-english
#   영어 전용 모델이므로 한국어 정확도가 낮음
# ──────────────────────────────────────────────
from transformers import pipeline

# 기본 pipeline 생성 (최초 실행 시 모델 자동 다운로드)
default_classifier = pipeline(task="sentiment-analysis")

# 테스트 문장 (한국어 + 영어 혼합)
inputs = [
    "이 영화 정말 좋았어요!",              # 한국어 긍정
    "거리에서 싸움이 나서 기분이 불쾌했어요",  # 한국어 부정
    "This movie was amazing!",              # 영어 긍정
    "I hated every minute of it.",          # 영어 부정
]

# 분류 실행
results = default_classifier(inputs)

# 결과 출력
print("=== 기본 pipeline (영어 전용 모델) 결과 ===\n")
for text, out in zip(inputs, results):
    print(f"  입력: {text}")
    print(f"    → label: {out['label']},  score: {out['score']:.4f}")
    print()

print("※ 한국어 문장의 결과가 부정확할 수 있습니다.")
print("  → 한국어 전용 모델을 지정하면 정확도가 크게 향상됩니다.")

In [ ]:
# ──────────────────────────────────────────────
# 감성 분석 (2): 한국어 모델 직접 지정
#   모델: sangrimlee/bert-base-multilingual-cased-nsmc
#   네이버 영화 리뷰(NSMC) 데이터로 미세조정된 BERT 모델
# ──────────────────────────────────────────────
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# ---- 한국어 감성 분석 모델 로드 ----
model_name = "sangrimlee/bert-base-multilingual-cased-nsmc"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# pipeline에 모델과 토크나이저를 직접 지정
korean_classifier = pipeline(
    task="sentiment-analysis",
    model=model,
    tokenizer=tokenizer
)

# 한국어 테스트 문장
inputs = [
    "이 영화 정말 좋았어요!",
    "거리에서 싸움이 나서 기분이 불쾌했어요",
]

# ---- top_k=None → 모든 클래스의 확률 반환 ----
#   참고: 과거의 return_all_scores=True 는 deprecated
#         top_k=None 으로 대체
outputs = korean_classifier(inputs, top_k=None)

# 라벨 매핑 확인
labels = model.config.id2label    # 예: {0: "negative", 1: "positive"}
print(f"라벨 매핑: {labels}\n")

# 결과 출력
print("=== 한국어 모델 + top_k=None 결과 ===\n")
for text, scores in zip(inputs, outputs):
    print(f"  입력: {text}")
    for item in scores:
        print(f"    → {item['label']:10s}: {item['score']:.4f}")
    print()

## 5. 예제 2: 번역 (Translation)

텍스트를 한 언어에서 다른 언어로 번역하는 태스크입니다.

| 구분 | 내용 |
|------|------|
| Task 이름 | `translation` |
| 사용 모델 | facebook/nllb-200-distilled-600M (NLLB) |
| 지원 언어 | 200개 이상 (한국어 포함) |
| 출력 | translation_text (번역된 텍스트) |

### NLLB (No Language Left Behind)

Meta(Facebook)에서 개발한 **다국어 번역 모델**로, **200개 이상의 언어**를 지원합니다.

| 모델 | 크기 | 특징 |
|------|------|------|
| nllb-200-distilled-600M | ~600MB | 가볍고 빠름 (추천) |
| nllb-200-1.3B | ~1.3GB | 더 높은 번역 품질 |
| nllb-200-3.3B | ~3.3GB | 최고 품질, GPU 권장 |

### 주요 언어 코드 (FLORES-200)

| 언어 | 코드 | 언어 | 코드 |
|------|------|------|------|
| 한국어 | `kor_Hang` | 영어 | `eng_Latn` |
| 일본어 | `jpn_Jpan` | 중국어(간체) | `zho_Hans` |
| 프랑스어 | `fra_Latn` | 독일어 | `deu_Latn` |

> **주의 — Transformers v5 사용자**:
> v5에서 `TranslationPipeline`이 제거되어 `pipeline("translation", ...)` 이 작동하지 않을 수 있습니다.
> v5에서는 `text-generation` pipeline + LLM 방식을 사용하거나,
> `pip install 'transformers>=4.45,<5'` 로 v4를 설치하여 아래 코드를 실행하세요.

In [ ]:
# ──────────────────────────────────────────────
# 번역: NLLB 모델로 한국어 ↔ 영어 번역
#   모델: facebook/nllb-200-distilled-600M
#   src_lang / tgt_lang 으로 언어 코드 지정
#   한국어: kor_Hang,  영어: eng_Latn
#   ※ Transformers v5에서는 TranslationPipeline 제거됨
#     v5 사용 시 pip install 'transformers<5' 필요
# ──────────────────────────────────────────────
from transformers import pipeline

# NLLB 다국어 번역 모델 로드 (최초 실행 시 ~600MB 다운로드)
translator = pipeline(
    "translation",
    model="facebook/nllb-200-distilled-600M"
)

# ---- 1) 영어 → 한국어 ----
text_en = "This movie is really interesting!"
result_en_ko = translator(
    text_en,
    src_lang="eng_Latn",     # 원본 언어: 영어
    tgt_lang="kor_Hang"      # 목표 언어: 한국어
)
print("=== 영어 → 한국어 ===")
print(f"  입력: {text_en}")
print(f"  번역: {result_en_ko[0]['translation_text']}")
print()

# ---- 2) 한국어 → 영어 ----
text_ko = "이 영화 정말 재미있어요!"
result_ko_en = translator(
    text_ko,
    src_lang="kor_Hang",     # 원본 언어: 한국어
    tgt_lang="eng_Latn"      # 목표 언어: 영어
)
print("=== 한국어 → 영어 ===")
print(f"  입력: {text_ko}")
print(f"  번역: {result_ko_en[0]['translation_text']}")
print()

# ---- 3) 한국어 → 일본어 ----
text_ko2 = "오늘 날씨가 정말 좋습니다."
result_ko_ja = translator(
    text_ko2,
    src_lang="kor_Hang",     # 원본: 한국어
    tgt_lang="jpn_Jpan"      # 목표: 일본어
)
print("=== 한국어 → 일본어 ===")
print(f"  입력: {text_ko2}")
print(f"  번역: {result_ko_ja[0]['translation_text']}")

## 6. 예제 3: 이미지 분류 (Image Classification)

이미지가 어떤 카테고리에 속하는지 분류하는 태스크입니다.

| 구분 | 내용 |
|------|------|
| Task 이름 | `image-classification` |
| 기본 모델 | google/vit-base-patch16-224 (Vision Transformer) |
| 비교 모델 | facebook/convnext-base-224 (ConvNeXt) |
| 학습 데이터 | ImageNet (1,000개 클래스) |
| 출력 | label (클래스명), score (확률) |

### 모델 비교

| 모델 | 아키텍처 | 특징 |
|------|----------|------|
| **ViT** (Vision Transformer) | Transformer | 이미지를 16x16 패치로 나누어 Transformer에 입력 |
| **ConvNeXt** | 모던 CNN | ResNet 구조를 Transformer 기법(LayerNorm 등)으로 개선 |

> **참고**: ImageNet 1,000 클래스에는 동물(개, 고양이, 새), 사물(가구, 차량),
> 풍경(산, 숲) 등 다양한 카테고리가 포함되어 있습니다.

In [ ]:
# ──────────────────────────────────────────────
# 이미지 분류 (1): 기본 pipeline — ViT 모델
#   기본 모델: google/vit-base-patch16-224
#   ImageNet 1,000 클래스 분류
# ──────────────────────────────────────────────
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

# 기본 이미지 분류 pipeline
image_classifier = pipeline(task="image-classification")

# ---- 테스트 이미지 로드 (URL → PIL Image) ----
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"

response = requests.get(image_url)
image = Image.open(BytesIO(response.content))

# 이미지 표시
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Input Image", fontsize=14)
plt.show()

# ---- 분류 실행 (기본: 상위 5개 결과 반환) ----
predictions = image_classifier(images=image)

print("=== 기본 pipeline (ViT) 분류 결과 ===\n")
for pred in predictions:
    print(f"  {pred['label']:30s}  score: {pred['score']:.4f}")

In [ ]:
# ──────────────────────────────────────────────
# 이미지 분류 (2): ConvNeXt 모델 지정
#   모델: facebook/convnext-base-224
#   AutoImageProcessor 사용 (AutoFeatureExtractor는 deprecated)
# ──────────────────────────────────────────────
from transformers import pipeline, AutoImageProcessor, AutoModelForImageClassification

# ---- 모델과 이미지 프로세서 로드 ----
model_name = "facebook/convnext-base-224"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

# pipeline에 모델과 프로세서 지정
#   image_processor= 사용 (feature_extractor= 는 deprecated)
custom_classifier = pipeline(
    task="image-classification",
    model=model,
    image_processor=processor
)

# ---- 동일 이미지로 분류 실행 ----
custom_predictions = custom_classifier(images=image)

print("=== ConvNeXt 모델 분류 결과 ===\n")
for pred in custom_predictions:
    print(f"  {pred['label']:30s}  score: {pred['score']:.4f}")

print("\n※ 같은 이미지라도 모델에 따라 분류 결과(순위·확률)가 다를 수 있습니다.")

## 7. 예제 4: 객체 탐지 (Object Detection)

이미지 안에서 **물체의 위치(바운딩 박스)**와 **종류(라벨)**를 동시에 찾는 태스크입니다.

| 구분 | 내용 |
|------|------|
| Task 이름 | `object-detection` |
| 기본 모델 | facebook/detr-resnet-50 (DETR) |
| 비교 모델 | facebook/detr-resnet-101 |
| 학습 데이터 | COCO (91개 클래스) |
| 출력 | label, score, box (xmin, ymin, xmax, ymax) |

### DETR (DEtection TRansformer)

Facebook AI Research에서 개발한 **End-to-End 객체 탐지 모델**입니다.

| 특징 | 설명 |
|------|------|
| 백본 (Backbone) | ResNet-50 또는 ResNet-101로 이미지 특징 추출 |
| 디코더 (Decoder) | Transformer로 객체 위치와 클래스를 직접 예측 |
| 앵커 프리 (Anchor-free) | 기존 객체 탐지의 앵커 박스가 불필요 |
| End-to-End | NMS(Non-Maximum Suppression) 없이 직접 예측 |

### COCO 데이터셋 주요 클래스 예시

사람(person), 자동차(car), 자전거(bicycle), 고양이(cat), 개(dog),
의자(chair), 소파(couch), TV(tv), 노트북(laptop), 핸드폰(cell phone) 등 **91개 클래스**

In [ ]:
# ──────────────────────────────────────────────
# 객체 탐지 결과 시각화 함수 (재사용)
#   바운딩 박스 + 라벨 + 확률을 이미지 위에 표시
# ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def show_detections(image, detections, title="Object Detection", threshold=0.7):
    """
    객체 탐지 결과를 이미지 위에 시각화합니다.

    Parameters:
        image      : PIL Image 객체
        detections : pipeline 출력 [{'label', 'score', 'box'}, ...]
        title      : 그래프 제목
        threshold  : 이 확률 이상인 객체만 표시 (기본값 0.7)
    """
    # 라벨별 다른 색상 배정
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4',
              '#FFEAA7', '#DDA0DD', '#98D8C8', '#F7DC6F']
    label_colors = {}
    color_idx = 0

    fig, ax = plt.subplots(1, figsize=(12, 9))
    ax.imshow(image)

    for det in detections:
        if det['score'] < threshold:
            continue

        label = det['label']
        score = det['score']
        box   = det['box']

        # 라벨별 색상 할당
        if label not in label_colors:
            label_colors[label] = colors[color_idx % len(colors)]
            color_idx += 1
        color = label_colors[label]

        # 바운딩 박스 그리기
        x0, y0 = box['xmin'], box['ymin']
        width  = box['xmax'] - box['xmin']
        height = box['ymax'] - box['ymin']

        rect = patches.Rectangle(
            (x0, y0), width, height,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)

        # 라벨 + 확률 텍스트
        ax.text(
            x0, y0 - 5,
            f"{label} ({score:.2f})",
            fontsize=11, color='white', fontweight='bold',
            bbox=dict(facecolor=color, alpha=0.7, pad=2)
        )

    ax.set_title(title, fontsize=14)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


print("show_detections() 함수 정의 완료")

In [ ]:
# ──────────────────────────────────────────────
# 객체 탐지 (1): 기본 pipeline — DETR-ResNet-50
#   기본 모델: facebook/detr-resnet-50
#   COCO 91 클래스 탐지
# ──────────────────────────────────────────────
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# 기본 객체 탐지 pipeline
detector = pipeline(task="object-detection")

# ---- 테스트 이미지 로드 ----
#   COCO 데이터셋 샘플 이미지 (고양이 2마리 + 리모컨)
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
response = requests.get(image_url)
det_image = Image.open(BytesIO(response.content))

# ---- 탐지 실행 ----
detections = detector(det_image)

# 텍스트 결과 출력
print("=== 기본 pipeline (DETR-ResNet-50) 탐지 결과 ===\n")
for det in detections:
    label = det['label']
    score = det['score']
    box   = det['box']
    print(f"  {label:15s}  score: {score:.4f}  box: {box}")

# 시각화
print()
show_detections(det_image, detections,
                title="DETR-ResNet-50 (기본 모델)", threshold=0.7)

In [ ]:
# ──────────────────────────────────────────────
# 객체 탐지 (2): DETR-ResNet-101 모델 지정
#   더 깊은 백본(ResNet-101)으로 정밀도 향상
#   AutoImageProcessor 사용 (DetrFeatureExtractor는 deprecated)
# ──────────────────────────────────────────────
from transformers import pipeline, AutoImageProcessor, AutoModelForObjectDetection

# ---- 모델과 이미지 프로세서 로드 ----
model_name = "facebook/detr-resnet-101"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForObjectDetection.from_pretrained(model_name)

# pipeline에 모델과 프로세서 지정
custom_detector = pipeline(
    task="object-detection",
    model=model,
    image_processor=processor
)

# ---- 동일 이미지로 탐지 실행 ----
custom_detections = custom_detector(det_image)

# 텍스트 결과 출력
print("=== DETR-ResNet-101 모델 탐지 결과 ===\n")
for det in custom_detections:
    label = det['label']
    score = det['score']
    box   = det['box']
    print(f"  {label:15s}  score: {score:.4f}  box: {box}")

# 시각화
print()
show_detections(det_image, custom_detections,
                title="DETR-ResNet-101 (모델 지정)", threshold=0.7)

print("\n※ ResNet-101은 ResNet-50보다 레이어가 깊어 더 정밀한 탐지가 가능합니다.")
print("  다만 모델 크기가 크고 추론 속도가 느릴 수 있습니다.")

---

## Transformers Pipeline 핵심 문법 요약 (Quick Reference)

### Pipeline 생성

| 방법 | 코드 | 설명 |
|------|------|------|
| Task만 지정 | `pipeline("sentiment-analysis")` | 기본 모델 자동 선택 |
| 모델 지정 | `pipeline("sentiment-analysis", model="...")` | 특정 모델 사용 |
| 분리 로드 | `pipeline(task=..., model=model, tokenizer=tokenizer)` | 세밀한 제어 |

### 주요 Pipeline Task

| Task | 설명 | 예시 코드 |
|------|------|-----------|
| `sentiment-analysis` | 감성 분석 | `pipe("좋은 영화!")` |
| `text-classification` | 텍스트 분류 | `pipe("입력 텍스트")` |
| `translation` | 번역 (v4) | `pipe(text, src_lang=..., tgt_lang=...)` |
| `text-generation` | 텍스트 생성 | `pipe("Once upon a time")` |
| `image-classification` | 이미지 분류 | `pipe(images=image)` |
| `object-detection` | 객체 탐지 | `pipe(image)` |

### 주요 Auto 클래스

| 클래스 | 용도 |
|--------|------|
| `AutoTokenizer` | 텍스트 토크나이저 자동 선택 |
| `AutoImageProcessor` | 이미지 전처리기 자동 선택 |
| `AutoModelForSequenceClassification` | 텍스트 분류 모델 |
| `AutoModelForImageClassification` | 이미지 분류 모델 |
| `AutoModelForObjectDetection` | 객체 탐지 모델 |

### 자주 사용하는 매개변수

| 매개변수 | 설명 | 예시 |
|----------|------|------|
| `task` | 수행할 태스크 이름 | `"sentiment-analysis"` |
| `model` | 모델 이름 또는 경로 | `"facebook/detr-resnet-50"` |
| `tokenizer` | 토크나이저 (텍스트용) | `AutoTokenizer.from_pretrained(...)` |
| `image_processor` | 이미지 전처리기 | `AutoImageProcessor.from_pretrained(...)` |
| `device` | 실행 디바이스 | `0` (GPU) 또는 `-1` (CPU) |
| `top_k` | 상위 k개 결과 반환 | `None` (전체), `5` (상위 5개) |

### Deprecated → 최신 API 변경 사항

| 기존 (Deprecated) | 최신 | 비고 |
|--------------------|------|------|
| `return_all_scores=True` | `top_k=None` | 전체 클래스 확률 반환 |
| `AutoFeatureExtractor` | `AutoImageProcessor` | 이미지 전처리기 |
| `DetrFeatureExtractor` | `AutoImageProcessor` | DETR 이미지 전처리기 |
| `feature_extractor=` | `image_processor=` | pipeline 매개변수명 |

> **Transformers v5 주요 변경사항** (2026.01):
> - `TranslationPipeline`, `SummarizationPipeline` 제거 → `text-generation` 사용 권장
> - `AutoFeatureExtractor` → `AutoImageProcessor` 통합
> - PyTorch 전용 백엔드 (TensorFlow/Flax 지원 축소)
> - 매주 마이너 버전 릴리스 (v5.1, v5.2, ...)